# Duygu Analizi Örneği

Bugün derste basit bir duygu analizi örneğini incelemiştik. Şimdi pratik bir kod ile örneğin adımlarını ele alacağız.

Herhangi bir analiz için alınacak kararlar:
- Giriş verileri
- Olası çıktılar
- Karar algoritması
- Değerlendirme yöntemi

Bu duygu analizi alıştırmasında, sözlük yöntemini kullanarak olumlu veya olumsuz olup olmadığına karar vermek için yalnızca metin verilerini kullanacağız. Sonunda, sistemimizi NLTK'nin movie_reviews veri kümesiyle doğruluk ölçütü kullanarak değerlendireceğiz.

In [ ]:
#import libraries
#download corpora

import nltk

nltk.download('movie_reviews')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('opinion_lexicon')


from nltk import word_tokenize
from nltk import punkt
from nltk.corpus import stopwords
from nltk.corpus import opinion_lexicon
from nltk.corpus import movie_reviews
import numpy as np


In [ ]:
#duygu skorunu ölçmek için bir metod
def evaluate(sentiment_predicted_labels, sentiment_gold_labels):
    '''
    type sentiment_predicted_labels: list
    param sentiment_predicted_labels: Tahmin yapılar etiketler
    type sentiment_gold_labels: list
    param sentiment_gold_labels: Gerçekte olan duygu durumu
    rtype: float
    return: sistem ne kadar doğru tahmin etmiş?
    '''

    predicted_np = np.array(sentiment_predicted_labels)
    gold_np = np.array(sentiment_gold_labels)

    matched = (predicted_np == gold_np) #tahmin edilen ve gerçekte olan duygu etiketi aynı! Ne kadar doğruysa bizim sistemimizin performansı da o kadar yüksek
    accuracy = matched.sum() / matched.size #toplamda kaç değerlendirmenin etiketi doğru bilinmiş.

    return accuracy

## Sınıflandırma için Denetimli Makine Öğrenimi

Verilen etiketlerden yola çıkarak bir sistem eğiteceğiz ve bu eğitilmiş sistemi yeni örnekler üzerinde kullanarak verilen metni sınıflandıracağız. Naive Bayes ve Lojistik Regresyon sınıflandırıcılarını öğrendik. Bunları kullanarak yorumları olumlu ve olumsuz olarak sınıflandıralım.

In [ ]:

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from collections import defaultdict
import math



X = [] #kelimelere ayrılmış cümleler
y = [] #veri etiketi, pozitif ya da negatif


for i in movie_reviews.fileids():
    label = i.split('/')[0]
    input_text = movie_reviews.raw(i)
    X.append(input_text)
    if label == "neg":
      y.append(0)
    else:
      y.append(1)

#veri setini eğitim, validasyon, ve test olarak üçe böleceğiz.

#önce test setini ayırıyoruz. Verinin yüzde 20si test olacak.
X_traindev, X_test, y_traindev, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

#Geri kalan data eğitim ve validasyona bölünecek.
X_train, X_dev, y_train, y_dev = train_test_split(X_traindev, y_traindev, test_size=0.25, random_state=1)




In [ ]:

vectorizer = CountVectorizer()

#kelimeleri sayıp dönüştürecek kod
X_train = vectorizer.fit_transform(X_train)
X_dev = vectorizer.transform(X_dev)



In [ ]:
# Naive Bayes Sınıflandırması
mnb = MultinomialNB()
mnb.fit(X_train,y_train)
y_predicted = mnb.predict(X_dev)

In [ ]:
print('Naive Bayes ile duygu analizi yapan aracın doğruluk ölçüsü: {:.2f}'.format(evaluate(y_predicted, y_dev)))

In [ ]:
log_regression = LogisticRegression(solver='lbfgs', max_iter=400)
log_regression.fit(X_train,y_train)
y_predicted = log_regression.predict(X_dev)

In [ ]:
print('Logistik regresyon ile duygu analizi yapan aracın doğruluk ölçüsü: {:.2f}'.format(evaluate(y_predicted, y_dev)))

In [ ]:
#Test verisinde son performans ölçümü
X_test = vectorizer.transform(X_test)
y_predicted_nb = mnb.predict(X_test)
print('Naive Bayes ile duygu analizi yapan aracın doğruluk ölçüsü: {:.2f}'.format(evaluate(y_predicted_nb, y_test)))
y_predicted_reg = log_regression.predict(X_test)
print('Logistik regresyon ile duygu analizi yapan aracın doğruluk ölçüsü: {:.2f}'.format(evaluate(y_predicted_reg, y_test)))


##Word2Vec

Kelime anlamlarının uzayda vektörler atanarak nasıl temsil edilebileceğini gördük. Bu temsillerden biri de kelimelerin yoğun kısa vektörlerle temsil edildiği Word2Vec'tir. Bu alıştırmada, NLTK'den kullanabileceğimiz Word2Vec eğitimli gömme vektörlerini inceleyeceğiz.

Word2Vec modelleri için Gensim kütüphanesini kullanacağız. Gensim, LDA gibi konu modelleme algoritmalarına da sahip kullanışlı bir kütüphanedir. Bu eğitim, https://www.nltk.org/howto/gensim.html adresindeki adımları takip eder.

In [ ]:
!pip install gensim
import gensim
from nltk.test.gensim_fixt import setup_module
setup_module()

### Önceden Eğitilmiş Modeller

Kullandığımız veri seti çok büyük değildi. Bu yüzden Word2Vec modelini kendimiz eğitmeyeceğiz. NLTK ayrıca 100 milyar kelime içeren Google Haberler Veri Seti üzerinde eğitilmiş önceden eğitilmiş bir Word2Vec modeli de sunuyor. Tüm kelimeleri içeren model yaklaşık 3 GB olduğundan, budanmış modeli kullanacağız.

In [ ]:
from nltk.data import find
nltk.download('word2vec_sample')
word2vec_sample = str(find('models/word2vec_sample/pruned.word2vec.txt'))
model = gensim.models.KeyedVectors.load_word2vec_format(word2vec_sample, binary=False)

In [ ]:
#around 44K words in the model
len(model)

In [ ]:
#each word is represented by 300 dimensions.
len(model['amazing'])

In [ ]:
#Verilen kelimeye en yakın 10 kelimeyi bulalım.
model.most_similar(positive=['amazing'], topn = 10)

In [ ]:
model.similarity('sun','swim')

In [ ]:
model.similarity('sun','beach')

In [ ]:
model.similarity('sun','moon')

Güneş ile daha yüksek bir skor veren kelime bulabilir misiniz? 2 dk deneyelim.

Bu kelime temsillerini kullarak temaya uymayan kelimeleri de bulabiliriz.

In [ ]:
#aşağıdaki kelimelerden hangisi diğerlerine uymuyor?
model.doesnt_match('breakfast cereal dinner lunch'.split())

Gömülü vektörler tarafından yakalanan kelimeler arasındaki anlamsal ilişkileri inceledik.


Kral - Erkek + Kadın = Kraliçe


Almanya - Berlin + Paris = Fransa

In [ ]:
model.most_similar(positive=['Man','King'], negative=['Woman'], topn = 1)

In [ ]:
model.most_similar(positive=['Paris','Germany'], negative=['Berlin'], topn = 1)

In [ ]:
model.most_similar(positive=['program','snake'], negative=['animal'], topn = 1)

Bu gömülü vektörleri kullanarak denemeler yapalım. Özellikle duygu analizi üzerinde bakabiliriz.